# Predicting Experimental koff Values from Protein-Ligand Complexes Using ProtBert and ChemBERTa Embeddings
 This notebook aims to predict experimental koff values for a dataset of protein-ligand complexes. To achieve this, we leverage ProtBERT embeddings for protein sequences and ChemBERTa embeddings for ligand representations.


In [ ]:
# Standard library
import os
import re
import copy
import json
import pickle
import random
import warnings
from typing import List

# Numerical & data handling
import numpy as np
import pandas as pd

# Machine Learning
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR

# Deep Learning
import torch
import torch.nn as nn
import torch.nn.functional as F

# Transformers
from transformers import BertModel, BertTokenizer, AutoTokenizer, AutoModel

# RDKit
from rdkit import Chem
from rdkit.Chem import inchi

# Gradient Boosting / Optimization
import xgboost as xgb
import optuna

# Statistics
from scipy.stats import pearsonr

# Utilities
from tqdm.auto import tqdm
from IPython.display import display

# Settings
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

## Software Environment

This notebook was developed and executed using the following Python environment and core scientific computing libraries:

- Python: 3.12.9
- PyTorch: 2.7.1
- NumPy: 2.2.6
- Pandas: 2.2.3
- Scikit-learn: 1.6.1


## ProtBERT Maximum Sequence Length Check

In [ ]:
model_name = "Rostlab/prot_bert"
tokenizer = BertTokenizer.from_pretrained(model_name, do_lower_case=False)
model = BertModel.from_pretrained(model_name)
print("Tokenizer max length:", tokenizer.model_max_length)
print("Model max position embeddings:", model.config.max_position_embeddings)

Tokenizer max length: 1000000000000000019884624838656
Model max position embeddings: 40000


## ChemBERTa Maximum Token Length Check

This cell loads the pretrained ChemBERTa model and prints its maximum number of positional embeddings, which defines the maximum token length used during SMILES embedding extraction.

In [4]:
model = AutoModel.from_pretrained("seyonec/ChemBERTa-zinc-base-v1")
print(model.config.max_position_embeddings)

514


## 1. Load Expanded Dataframe

In [ ]:
# Load expanded dataset from csv file
df_annot = pd.read_csv('dataset.csv')
df_annot
# normalization protein_type
def normalize_protein_name(x):
    if pd.isna(x):
        return x

    x = str(x).strip()

    # Standardize greek letters, hyphens, spaces and case
    x = x.replace("α", "ALPHA")
    x = x.replace("-", " ")
    x = re.sub(r"\s+", " ", x)
    x = x.upper()

    # Unify known synonyms / variants
    synonyms = {
        # HSP90
        "HEAT SHOCK PROTEIN HSP 90 ALPHA": "HSP90",
        "HSP90 ALPHA": "HSP90",

        # p38
        "P38Α": "P38 ALPHA",
        "P38A": "P38 ALPHA",
        "P38B0": "P38 ALPHA",

        # Adenosine A2A receptor
        "A2A": "ADENOSINE A2A RECEPTOR",
        "A2A ADENOSINE RECEPTOR A2AR STAR2 BRIL": "ADENOSINE A2A RECEPTOR",
        "THERMOSTABILISED ADENOSINE A2A RECEPTOR": "ADENOSINE A2A RECEPTOR",

        # Soluble epoxide hydrolase
        "SOLUBLE EPOXIDE HYDROLASE": "SOLUBLE EPOXIDE HYDROLASE",
        "SEH": "SOLUBLE EPOXIDE HYDROLASE",
    }

    return synonyms.get(x, x)
# apply normalization
df_annot["protein_type"] = df_annot["protein_type"].apply(normalize_protein_name)
df_annot


,smiles,canonical_smiles,koff,protbert_sequence,protein_sequence,inchikey,protein_type,pdb_path
0,O=C(Nc1cc(Nc2nc(-c3cnccc3)ccn2)c(C)cc1)c1ccc(C...,Cc1ccc(NC(=O)c2ccc(C[N@H+]3CC[N@@H+](C)CC3)cc2...,0.380000,R P T F Y R Q E L N K T I W E V P E R Y Q N L ...,RPTFYRQELNKTIWEVPERYQNLSPVGSGAYGSVCAAFDTKTGLRV...,KTUFNOKKBVMGRW-UHFFFAOYSA-P,P38ALPHA,3hec_ligand_native_1
1,Clc1c(C(F)(F)F)cc(NC(=O)Nc2ccc(Oc3cc(C(=O)NC)n...,CNC(=O)c1cc(Oc2ccc(NC(=O)Nc3ccc(Cl)c(C(F)(F)F)...,0.018000,R P T F Y R Q E L N K T I W E V P E R Y Q N L ...,RPTFYRQELNKTIWEVPERYQNLSPVGSGAYGSVCAAFDTKTGLRV...,MLDQJTXFUGDVEO-UHFFFAOYSA-N,P38ALPHA,3heg_ligand_native_2
2,O=C(Nc1n(-c2ccc(C)cc2)nc(C(C)(C)C)c1)Nc1c2c(c(...,Cc1ccc(-n2nc(C(C)(C)C)cc2NC(=O)Nc2ccc(OCC[NH+]...,0.000008,M S Q E R P T F Y R Q E L N K T I W E V P E R ...,MSQERPTFYRQELNKTIWEVPERYQNLSPVGSGAYGSVCAAFDTKT...,MVCOAUNKQVWQHZ-UHFFFAOYSA-O,P38ALPHA,1kv2_ligand_native_3
3,Clc1ccc(NC(=O)Nc2n(C)nc(C(C)(C)C)c2)cc1,Cn1nc(C(C)(C)C)cc1NC(=O)Nc1ccc(Cl)cc1,0.062000,M S Q E R P T F Y R Q E L N K T I W E V P E R ...,MSQERPTFYRQELNKTIWEVPERYQNLSPVGSGAYGSVCAAFDTKT...,FWIJKWMXNHRSRO-UHFFFAOYSA-N,P38ALPHA,1kv1_ligand_native_4
4,O=C(Nc1n(-c2cc(C)ccc2)nc(C(C)(C)C)c1)Nc1ccc(Nc...,Cc1cccc(-n2nc(C(C)(C)C)cc2NC(=O)Nc2ccc(Nc3ncnc...,0.001695,S Q E R P T F Y R Q E L N K T I W E V P E R Y ...,SQERPTFYRQELNKTIWEVPERYQNLSPVGSGAYGSVCAAFDTKTG...,ZKESOLNZMJUNTF-UHFFFAOYSA-N,P38ALPHA,3gcq_ligand_native_5
...,...,...,...,...,...,...,...,...
1067,CCCCCN(C)C(=O)C(=O)c3c(c2ccc1ccccc1c2)[nH]c4cc...,CCCCCN(C)C(=O)C(=O)c1c(-c2ccc3ccccc3c2)[nH]c2c...,0.000117,M P E S W V P A V G L T L V P S L G G F M G A ...,MPESWVPAVGLTLVPSLGGFMGAYFVRGEGLRWYAGLQKPSWHPPR...,QYVXQRCNMDJGLL-UHFFFAOYSA-N,TSPO,structures/2mgy/2mgy_prot_1.pdb
1068,CCCN(CCC)C(=O)C(=O)c2c(c1ccc(C)cc1)[nH]c3ccccc23,CCCN(CCC)C(=O)C(=O)c1c(-c2ccc(C)cc2)[nH]c2ccccc12,0.000150,M P E S W V P A V G L T L V P S L G G F M G A ...,MPESWVPAVGLTLVPSLGGFMGAYFVRGEGLRWYAGLQKPSWHPPR...,WCIYVVCNCVAWIC-UHFFFAOYSA-N,TSPO,structures/2mgy/2mgy_prot_1.pdb
1069,CCC(C)N(C)C(=O)c3cc1ccccc1c(c2ccccc2Cl)n3,CCC(C)N(C)C(=O)c1cc2ccccc2c(-c2ccccc2Cl)n1,0.000483,M P E S W V P A V G L T L V P S L G G F M G A ...,MPESWVPAVGLTLVPSLGGFMGAYFVRGEGLRWYAGLQKPSWHPPR...,RAVIZVQZGXBOQO-UHFFFAOYSA-N,TSPO,structures/2mgy/2mgy_prot_1.pdb
1070,CN3C(=O)C/N=C(c1ccc(Cl)cc1)\c2cc(Cl)ccc23,CN1C(=O)CN=C(c2ccc(Cl)cc2)c2cc(Cl)ccc21,0.000517,M P E S W V P A V G L T L V P S L G G F M G A ...,MPESWVPAVGLTLVPSLGGFMGAYFVRGEGLRWYAGLQKPSWHPPR...,PUMYFTJOWAJIKF-UHFFFAOYSA-N,TSPO,structures/2mgy/2mgy_prot_1.pdb


## 2. Koff Preprocessing and Outlier Removal


In [6]:
# Remove koff outliers with IQR method:
# Calculate IQR for koff values
Q1 = df_annot['koff'].quantile(0.25)
Q3 = df_annot['koff'].quantile(0.75)
IQR = Q3 - Q1
# Define lower and upper bounds for outliers
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
# Identify outliers (before removing them)
outliers = df_annot[(df_annot['koff'] < lower_bound) |  (df_annot['koff'] > upper_bound)]
print(f"Number of removal outliers: {len(outliers)}")
display(outliers)
# Remove outliers
df_annot = df_annot[(df_annot['koff'] >= lower_bound) & (df_annot['koff'] <= upper_bound)]
# Check for NaNs in koff column:
df_annot['koff'].isna().sum()
# Convert to -log10(koff)
df_annot['koff'] = -np.log10(df_annot['koff'])
# Check for NaNs in koff column:
df_annot['koff'].isna().sum()
# Print those entries:
df_annot[df_annot['koff'].isna()]

Number of removal outliers: 185


,smiles,canonical_smiles,koff,protbert_sequence,protein_sequence,inchikey,protein_type,pdb_path
0,O=C(Nc1cc(Nc2nc(-c3cnccc3)ccn2)c(C)cc1)c1ccc(C...,Cc1ccc(NC(=O)c2ccc(C[N@H+]3CC[N@@H+](C)CC3)cc2...,0.380000,R P T F Y R Q E L N K T I W E V P E R Y Q N L ...,RPTFYRQELNKTIWEVPERYQNLSPVGSGAYGSVCAAFDTKTGLRV...,KTUFNOKKBVMGRW-UHFFFAOYSA-P,P38ALPHA,3hec_ligand_native_1
37,P(=O)([O-])(NC(C(=O)NCC)CC(C)C)CNC(=O)OCc1ccccc1,CCNC(=O)[C@H](CC(C)C)N[P@](=O)([O-])CNC(=O)OCc...,0.615000,I T G T S T V G V G R G V L G D Q K N I N T T ...,ITGTSTVGVGRGVLGDQKNINTTYSTYYYLQDNTRGNGIFTYDAKY...,AYDYEZVQGYFZKJ-HNNXBMFYSA-M,THERMOLYSIN,5lwd_ligand_7_45_dock
38,P(=O)([O-])(NC(C(=O)NCCC)CC(C)C)CNC(=O)OCc1ccccc1,CCCNC(=O)[C@H](CC(C)C)N[P@](=O)([O-])CNC(=O)OC...,0.171000,I T G T S T V G V G R G V L G D Q K N I N T T ...,ITGTSTVGVGRGVLGDQKNINTTYSTYYYLQDNTRGNGIFTYDAKY...,GSTDCDKICCYPKA-INIZCTEOSA-M,THERMOLYSIN,5lwd_ligand_8_46_dock
39,P(=O)([O-])(NC(C(=O)NCC(C)C)CC(C)C)CNC(=O)OCc1...,CC(C)CNC(=O)[C@H](CC(C)C)N[P@](=O)([O-])CNC(=O...,0.234000,I T G T S T V G V G R G V L G D Q K N I N T T ...,ITGTSTVGVGRGVLGDQKNINTTYSTYYYLQDNTRGNGIFTYDAKY...,CPLUMPZGNQOEMH-KRWDZBQOSA-M,THERMOLYSIN,5lwd_ligand_9_47_dock
40,P(=O)([O-])(NC(C(=O)NCC(C)(C)C)CC(C)C)CNC(=O)O...,CC(C)C[C@H](N[P@](=O)([O-])CNC(=O)OCc1ccccc1)C...,0.208000,I T G T S T V G V G R G V L G D Q K N I N T T ...,ITGTSTVGVGRGVLGDQKNINTTYSTYYYLQDNTRGNGIFTYDAKY...,GRURGYQNFDMDDQ-KRWDZBQOSA-M,THERMOLYSIN,5lwd_ligand_10_48_dock
...,...,...,...,...,...,...,...,...
975,C1CCN(C1)CC#CCN2CCCC2=O,O=C1CCCN1CC#CCN1CCCC1,0.293333,T I W Q V V F I A F L T G F L A L V T I I G N ...,TIWQVVFIAFLTGFLALVTIIGNILVIVAFKVNKQLKTVNNYFLLS...,RSDOPYMFZBJHRL-UHFFFAOYSA-N,MUSCARINIC M3,structures/5zhp/5zhp_prot_1.pdb
978,O=C(Nc1cc(nn1C)C(C)(C)C)Nc1ccc(cc1)Cl,Cn1nc(C(C)(C)C)cc1NC(=O)Nc1ccc(Cl)cc1,0.140000,R P T F Y R Q E L N K T I W E V P E R Y Q N L ...,RPTFYRQELNKTIWEVPERYQNLSPVGSGAYGSVCAAFDTKTGLRV...,FWIJKWMXNHRSRO-UHFFFAOYSA-N,P38 MAP KINASE,structures/1kv1/1kv1_prot_1.pdb
979,Cc2ccc(n1nc(C(C)(C)C)cc1NC(N)=O)cc2,Cc1ccc(-n2nc(C(C)(C)C)cc2NC(N)=O)cc1,0.140000,R P T F Y R Q E L N K T I W E V P E R Y Q N L ...,RPTFYRQELNKTIWEVPERYQNLSPVGSGAYGSVCAAFDTKTGLRV...,PXDGOKVFTJDAPC-UHFFFAOYSA-N,P38 MAP KINASE,structures/1kv1/1kv1_prot_1.pdb
994,FC1=CC=C(C=C1)C=1N=C(NC1C1=CC=NC=C1)C1=CC=C(C=...,CS(=O)c1ccc(-c2nc(-c3ccc(F)cc3)c(-c3ccncc3)[nH...,0.218000,H S Q E R P T F Y R Q E L N K T I W E V P E R ...,HSQERPTFYRQELNKTIWEVPERYQNLSPVGSGAYGSVCAAFDTKT...,CDMGBJANTYXAIV-UHFFFAOYSA-N,P38 MAP KINASE,structures/3zs5/3zs5_prot_1.pdb


,smiles,canonical_smiles,koff,protbert_sequence,protein_sequence,inchikey,protein_type,pdb_path


## 3. Stratified Train–Test Split Based on Koff Distribution

This cell creates the **training and test datasets** while preserving the distribution of the target variable.

First, the values of **−log10(koff)** are discretized into several bins, which approximate the distribution of the target variable.  
These bins are used to perform **stratified sampling**, ensuring that both the training and test sets maintain a similar distribution of koff values.

An **80/20 split** is applied using `train_test_split`, stratifying on the computed koff bins.

In [ ]:
# Split the dataset into training and test sets, with a 80/20 split, stratify based on label (koff) distribution:
# Calculate distribution of koff values and categorize into 10 bins
koff_bins = pd.cut(df_annot['koff'], bins=10, labels=False)
# Print the number of entries in each bin:
print(koff_bins.value_counts())
# Write bins into dataframe:
df_annot['koff_bin'] = koff_bins
# Split the dataset into training and test sets, with a 80/20 split, stratify based on label (koff) distribution:
df_train, df_test = train_test_split(df_annot, test_size=0.2, random_state=42, stratify=df_annot['koff_bin'])
# Save the training and test sets to csv files
df_train.to_csv('train_set.csv', index=False)
df_test.to_csv('test_set.csv', index=False)


koff
3    182
2    167
1    151
4    142
0    130
5     65
6     33
7     11
8      4
9      2
Name: count, dtype: int64


## 4. Generate protein and ligand embeddings using PROTBert and ChemBerta

In [8]:
# Re-load the training and test sets from csv files
df_train = pd.read_csv('train_set.csv')
df_test = pd.read_csv('test_set.csv')

# Protein Embedding Extraction with ProtBERT

In this step, we compute contextual protein embeddings using the **ProtBERT** model (`Rostlab/prot_bert`).

### What this cell does:

1. **Reads protein sequences** from the `protbert_sequence` column of `df_train`.
   - The row order is strictly preserved to guarantee alignment with the dataset.
   - Each sequence is normalized to be ProtBERT-compatible.

2. **Prepares input format for ProtBERT**
   - Amino acids are tokenized in the space-separated format expected by ProtBERT.
   - Non-standard characters are replaced with `"X"`.
   - The maximum allowed sequence length is derived directly from the model configuration (`max_position_embeddings`) after reserving positions for special tokens.
   - In this dataset, protein sequences are shorter than the model limit, so they are processed as full-length inputs without chunking.

3. **Computes embeddings**
   - For each sequence:
     - The full protein sequence is passed through ProtBERT in a single forward pass.
     - Mean pooling is applied across residue token embeddings, excluding `[CLS]` and `[SEP]`.
     - This produces one fixed-length vector per protein while preserving full-sequence context.
   - Final embedding dimension: **1024** per protein.

4. **Alignment safety**
   - The number of generated embeddings is checked against the number of dataset rows.
   - Order consistency is guaranteed (no shuffling).

5. **Outputs**
   - The embeddings are appended to `df_train` as `prot_embeddings`.
   - The full dataframe is saved as a `.parquet` file.
   - The embedding matrix `(N, 1024)` is also saved separately as a NumPy `.npy` file.


### Result

Each training sample is now enriched with a dense 1024-dimensional representation capturing the protein’s contextual biochemical features, ready for downstream machine learning models.


In [ ]:
# ProtBERT embeddings from df_train["protbert_sequence"] (order preserved)
# + save Parquet (df + embeddings) AND NumPy (embeddings only)

# PARAMS
MODEL_NAME = "Rostlab/prot_bert"   # hidden size = 1024
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
MAX_TOKENS = None                  # set from model.config.max_position_embeddings after loading
SEQ_COL    = "protbert_sequence"
AA_ALLOWED = set(list("ACDEFGHIKLMNPQRSTVWYX"))

def normalize_protbert_input(seq_raw: str) -> str:
    """
    Accepts:
    - already spaced sequences
    - non spaced sequences ('PQIT...')
    - sequences containing chains separated by '|' (these are joined using 'X')
    Returns a space-separated string ready for ProtBERT input.
    """
    if seq_raw is None or (isinstance(seq_raw, float) and np.isnan(seq_raw)):
        return ""

    s = str(seq_raw).strip().upper()
    if not s:
        return ""

    # Management of "AAA|BBB" chains (if any)
    s = s.replace("|", " X ")

    # If it is already spaced, process it by tokens; otherwise, process it by characters
    if " " in s:
        toks = [t for t in s.split() if t]
    else:
        toks = list(s)

    # Filter out invalid characters -> X
    toks = [(t if t in AA_ALLOWED else "X") for t in toks]

    return " ".join(toks)

@torch.no_grad()
def embed_protein(
    spaced_seq: str,
    tokenizer: BertTokenizer,
    model: BertModel,
    device: str,
    max_tokens: int
) -> np.ndarray:
    """
    Embed a full protein sequence without chunking.
    Mean pooling is applied over residue token embeddings, excluding [CLS] and [SEP].
    """
    hidden = model.config.hidden_size
    if not spaced_seq:
        return np.zeros((hidden,), dtype=np.float32)

    n_residue_tokens = len(spaced_seq.split())
    if n_residue_tokens > max_tokens:
        raise ValueError(
            f"Protein sequence has {n_residue_tokens} residue tokens, exceeding the model limit of {max_tokens}."
        )

    enc = tokenizer(
        spaced_seq,
        return_tensors="pt",
        add_special_tokens=True,
        padding=False,
        truncation=False
    )
    ids = enc["input_ids"].to(device)
    mask = enc["attention_mask"].to(device)

    if ids.size(1) > model.config.max_position_embeddings:
        raise ValueError(
            f"Tokenized sequence length {ids.size(1)} exceeds max_position_embeddings "
            f"({model.config.max_position_embeddings})."
        )

    out = model(input_ids=ids, attention_mask=mask).last_hidden_state  # [1,T,H]
    L = int(mask[0].sum().item())

    if L <= 2:
        vec = torch.zeros(hidden, device=out.device)
    else:
        vec = out[0, 1:L-1, :].mean(dim=0)  # exclude CLS/SEP

    return vec.detach().cpu().numpy().astype(np.float32)

# Sanity checks
if SEQ_COL not in df_train.columns:
    raise KeyError(f"Column '{SEQ_COL}' not found. Columns: {list(df_train.columns)}")

print(f"Load ProtBERT: {MODEL_NAME} (device={DEVICE})")
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME, do_lower_case=False)
model = BertModel.from_pretrained(MODEL_NAME).to(DEVICE).eval()

MAX_TOKENS = model.config.max_position_embeddings - tokenizer.num_special_tokens_to_add(pair=False)
print(
    f"ProtBERT max residue tokens: {MAX_TOKENS} "
    f"(max_position_embeddings={model.config.max_position_embeddings})"
)

# Compute embeddings in the SAME ORDER as df_train rows
seqs = df_train[SEQ_COL].tolist()
embs = []

for seq in tqdm(seqs, desc="ProtBERT embeddings", total=len(seqs)):
    spaced = normalize_protbert_input(seq)
    emb = embed_protein(spaced, tokenizer, model, DEVICE, MAX_TOKENS)
    embs.append(emb)

train_prot_embeddings = np.stack(embs, axis=0).astype(np.float32)

# L2 Normalization of ProtBERT embeddings
EPS = 1e-12
print("Applying L2 normalization to ProtBERT embeddings...")

norms = np.linalg.norm(train_prot_embeddings, axis=1, keepdims=True)
mask = norms > 0
train_prot_embeddings = np.where(
    mask,
    train_prot_embeddings / (norms + EPS),
    train_prot_embeddings
)

print("Normalization complete.")
print("embedding shape:", train_prot_embeddings.shape)
print(
    "Mean norm after normalization:",
    np.mean(np.linalg.norm(train_prot_embeddings, axis=1))
)

# Safety check: alignment
assert train_prot_embeddings.shape[0] == len(df_train), \
    "Mismatch between N embedding and N rowe df_train!"

print(f"Final embeddings shape: {train_prot_embeddings.shape}")

# Save embeddings as NumPy (.npy)
NPY_PATH = "train_prot_embeddings.npy"
np.save(NPY_PATH, train_prot_embeddings)
print(f"Embeddings matrix saved to: {NPY_PATH}")
# Append embeddings to df
df_train["prot_embeddings"] = train_prot_embeddings.tolist()
# Save full dataframe as Parquet
PARQUET_PATH = "train_with_prot_embeddings.parquet"

try:
    df_train.to_parquet(PARQUET_PATH, index=False, engine="pyarrow")
    print(f"Dataframe with embeddings saved to: {PARQUET_PATH}")
except Exception as e:
    print("Parquet save failed.")
    print("Reason:", repr(e))
    print("The .npy file was saved correctly.")

Carico ProtBERT: Rostlab/prot_bert (device=cpu)
ProtBERT max residue tokens: 39998 (max_position_embeddings=40000)


ProtBERT embeddings:   0%|          | 0/709 [00:00<?, ?it/s]

Applying L2 normalization to ProtBERT embeddings...
Normalization complete.
embedding shape: (709, 1024)
Mean norm after normalization: 1.0
Final embeddings shape: (709, 1024)
Embeddings matrix saved to: train_prot_embeddings.npy
Dataframe with embeddings saved to: train_with_prot_embeddings.parquet


# Ligand Representation Learning with ChemBERTa

In this step, we compute contextual molecular embeddings for each ligand using the pretrained transformer model **ChemBERTa** (`seyonec/ChemBERTa-zinc-base-v1`).

### Objective
To generate a fixed-length numerical representation (768-dimensional vector) for each ligand SMILES string, preserving the exact row order of the training dataset.

---

##  Processing Pipeline

1. **SMILES Extraction**
   - The canonical SMILES strings are read from `df_train["canonical_smiles"]`.
   - The original row order is strictly preserved.
   - Missing or empty SMILES are detected and handled safely.

2. **Model Inference**
   - ChemBERTa tokenizes each SMILES string.
   - Embeddings are computed in batches for efficiency.
   - Mean pooling (or CLS pooling) is applied to obtain a single 768-dimensional vector per molecule.

3. **Order Preservation**
   - Embeddings are reinserted into their original row positions.
   - Invalid or empty SMILES receive a zero vector.
   - This guarantees perfect alignment with the dataset.

4. **Optional Normalization**
   - L2 normalization can be applied to standardize embedding magnitude.
   - Zero vectors remain unchanged.

5. **Output**
   - The embeddings are appended to the dataframe as `ligand_embeddings`.
   - The full dataset is saved in Parquet format for efficient downstream ML usage.

---

##  Result

Each training sample is enriched with a 768-dimensional ligand representation capturing learned chemical structure information, ready to be combined with protein embeddings for binding kinetics modeling.

In [ ]:

# ChemBERTa embeddings from df_train[SMILES_COLUMN] (order preserved)
# + save Parquet (df + embeddings)

# PARAMS
SMILES_COLUMN = "canonical_smiles"
MODEL_NAME    = "seyonec/ChemBERTa-zinc-base-v1"  # hidden size = 768
DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE    = 64
MAX_LENGTH    = 514
POOLING       = "mean"   # "mean" or "cls"
NORMALIZE     = True     # L2 normalize
PARQUET_PATH  = "train_with_embeddings.parquet"

AA_EPS = 1e-12

# Sanity check
if "df_train" not in globals():
    raise NameError("df_train not found in memory. Load the dataframe first (df_train = ...).")

if SMILES_COLUMN not in df_train.columns:
    raise KeyError(f"Column '{SMILES_COLUMN}' not found. Columns: {list(df_train.columns)}")

if "prot_embeddings" not in df_train.columns:
    raise KeyError("Missing 'prot_embeddings'. Run ProtBERT embedding cell first.")


@torch.no_grad()
def embed_batch(
    texts: List[str],
    tokenizer: AutoTokenizer,
    model: AutoModel,
    device: str,
    max_length: int,
    pooling: str = "mean",
) -> np.ndarray:
    enc = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    )
    enc = {k: v.to(device) for k, v in enc.items()}
    out = model(**enc)

    last_hidden = out.last_hidden_state   # (B, T, H)
    input_ids   = enc["input_ids"]        # (B, T)
    attn        = enc["attention_mask"]   # (B, T)

    if pooling == "mean":
        # mask of special tokens + padding
        special_mask = torch.zeros_like(input_ids, dtype=torch.bool)

        if tokenizer.cls_token_id is not None:
            special_mask |= (input_ids == tokenizer.cls_token_id)
        if tokenizer.sep_token_id is not None:
            special_mask |= (input_ids == tokenizer.sep_token_id)
        if tokenizer.pad_token_id is not None:
            special_mask |= (input_ids == tokenizer.pad_token_id)

        # keep only real non-special tokens
        valid_mask = ((attn == 1) & (~special_mask)).unsqueeze(-1)  # (B, T, 1)

        summed = (last_hidden * valid_mask).sum(dim=1)              # (B, H)
        lengths = valid_mask.sum(dim=1).clamp(min=1)                # (B, 1)
        pooled = summed / lengths

    elif pooling == "cls":
        pooled = last_hidden[:, 0, :]                               # (B, H)

    else:
        raise ValueError("POOLING must be 'mean' or 'cls'")

    return pooled.detach().cpu().numpy().astype(np.float32)

# 1) Prepare SMILES preserving order
smiles_raw = df_train[SMILES_COLUMN].tolist()
N = len(smiles_raw)

smiles_str = [
    "" if (s is None or (isinstance(s, float) and np.isnan(s))) else str(s)
    for s in smiles_raw
]

is_valid = [(s.strip() != "") and (s.strip().lower() != "nan") for s in smiles_str]
valid_idx = [i for i, ok in enumerate(is_valid) if ok]
valid_texts = [smiles_str[i].strip() for i in valid_idx]

print(f"Total row: {N}")
print(f" SMILES: {len(valid_texts)} | empty SMILES: {N - len(valid_texts)}")

# 2) Load model
print(f"Load ChemBERTa: {MODEL_NAME} (device={DEVICE})")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE).eval()
H = model.config.hidden_size  # 768

# 3) Compute embeddings for valid only
vecs_valid = []
for i in tqdm(range(0, len(valid_texts), BATCH_SIZE), desc="ChemBERTa embeddings"):
    batch = valid_texts[i:i + BATCH_SIZE]
    v = embed_batch(batch, tokenizer, model, DEVICE, max_length=MAX_LENGTH, pooling=POOLING)
    vecs_valid.append(v)

X_valid = np.vstack(vecs_valid) if len(vecs_valid) > 0 else np.zeros((0, H), dtype=np.float32)

# Reinsert into original order, invalid = zero vectors
X = np.zeros((N, H), dtype=np.float32)
for j, i_orig in enumerate(valid_idx):
    X[i_orig, :] = X_valid[j, :]

# 4) Optional L2 normalization (leave zero-vectors unchanged)
if NORMALIZE:
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    X = np.where(norms > 0, X / (norms + AA_EPS), X)

print(f"Final ChemBERTa embeddings shape: {X.shape}")

# 5) Append to df + Save Parquet
df_train["ligand_embeddings"] = X.tolist()

if "chemberta_embeddings" in df_train.columns:
    df_train = df_train.drop(columns=["chemberta_embeddings"])

df_train.to_parquet(PARQUET_PATH, index=False, engine="pyarrow")
print(f"Dataframe with ChemBERTa embeddings saved to: {PARQUET_PATH}")


Total row: 709
 SMILES: 709 | empty SMILES: 0
Load ChemBERTa: seyonec/ChemBERTa-zinc-base-v1 (device=cpu)


ChemBERTa embeddings:   0%|          | 0/12 [00:00<?, ?it/s]

Final ChemBERTa embeddings shape: (709, 768)
Dataframe with ChemBERTa embeddings saved to: train_with_embeddings.parquet


## 5. Use the encoded complexes for NESTED CV

### Data Preparation and Feature Construction

In this step, the training dataset is loaded from a Parquet file containing precomputed protein and ligand embeddings. 
Basic integrity checks are performed to ensure that the required columns are present and that the embeddings are stored as NumPy arrays.

The protein and ligand embeddings are extracted and converted into numerical matrices. 
Any missing or infinite values are handled by replacing them with finite values to ensure numerical stability during model training.

The final input feature matrix is obtained by concatenating ligand and protein embeddings for each sample, producing a unified representation used as input for machine learning models. 
The target variable *p(koff)* is also extracted and converted into a NumPy array.

Finally, the shapes of the feature matrices and target vector are printed to verify consistency.

In [ ]:
df_train = pd.read_parquet("train_with_embeddings.parquet")

#embeddings numpy format
assert isinstance(df_train["prot_embeddings"].iloc[0], np.ndarray)

PROT_COL = "prot_embeddings"
LIG_COL  = "ligand_embeddings"
Y_COL    = "koff"

for c in [PROT_COL, LIG_COL, Y_COL]:
    if c not in df_train.columns:
        raise KeyError(f"Column '{c}' not found. Available Columns : {list(df_train.columns)}")

y_all = df_train[Y_COL].to_numpy(dtype=np.float32)
N = len(df_train)

prot = np.vstack(df_train[PROT_COL].values).astype(np.float32)
lig  = np.vstack(df_train[LIG_COL].values).astype(np.float32)

prot = np.nan_to_num(prot, nan=0.0, posinf=1e6, neginf=-1e6)
lig  = np.nan_to_num(lig,  nan=0.0, posinf=1e6, neginf=-1e6)

# concatenation [lig | prot]
X_all = np.concatenate([lig, prot], axis=1).astype(np.float32)
print("Shapes -> lig:", lig.shape, "| prot:", prot.shape, "| X:", X_all.shape, "| y:", y_all.shape)

Shapes -> lig: (709, 768) | prot: (709, 1024) | X: (709, 1792) | y: (709,)


### Nested Cross-Validation Framework for Model Comparison

This section implements a nested cross-validation pipeline to compare multiple regression models on the concatenated protein–ligand embedding space. Reproducibility is ensured by fixing the random seed across NumPy, Python, and PyTorch, while the available computation device (GPU or CPU) is automatically selected.

Several utility functions are defined to compute evaluation metrics, including RMSE, \(R^2\), and Pearson correlation, and to generate stratified bins from the continuous target variable for stratified splitting. In addition, a customizable multilayer perceptron (MLP) regressor is implemented in PyTorch, supporting different numbers of hidden layers, hidden dimensions, normalization strategies, activation functions, dropout, gradient clipping, Huber loss, and early stopping.

The core of the pipeline is the `objective` function, which defines the hyperparameter search space for each model and evaluates candidate configurations through inner cross-validation. The models considered in this framework are:
- **MLP**
- **XGBoost**
- **Support Vector Regressor (SVR)**
- **Random Forest (RF)**

For each outer fold, the training portion is further split into inner folds, and Optuna is used to optimize model-specific hyperparameters by minimizing the average validation RMSE across the inner splits. After selecting the best hyperparameter configuration, the model is retrained on the outer training set and evaluated on the corresponding outer test fold.

For the MLP model, an additional internal validation split is used within the training data to perform early stopping and estimate the optimal number of epochs. After this estimate is obtained, the MLP is retrained on the full available training fold for the selected number of epochs, and predictions are generated on the corresponding validation or outer test fold.

During the nested cross-validation process, performance metrics, optimal hyperparameters, and outer-fold predictions are collected and saved to disk both model-wise and globally. Finally, a summary table is generated by aggregating the mean and standard deviation of RMSE, \(R^2\), and Pearson correlation across outer folds, providing a robust comparison of model performance.

In [ ]:
RANDOM_STATE = 42
OUTER_SPLITS = 10
INNER_SPLITS = 10
N_TRIALS = 20
# Select GPU if available, otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Set random seeds for reproducibility across numpy, random, and torch
def set_seed(seed=42):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# Compute Root Mean Squared Error (RMSE)
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

# Compute Pearson correlation coefficient between true and predicted values
def pearson_corr(a, b):
    a = np.asarray(a).reshape(-1)
    b = np.asarray(b).reshape(-1)
    if len(a) < 2:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])

# Compute evaluation metrics (RMSE, R2, Pearson) and return as a dictionary
def compute_metrics(y_true, y_pred):
    return {
        "rmse": rmse(y_true, y_pred),
        "r2": r2_score(y_true, y_pred),
        "pearson": pearson_corr(y_true, y_pred)
    }

# Create stratified bins from a continuous target using quantiles
def make_strat_bins(y, q=10):
    """
    Create stratified bins for continue for continuous variables
    """
    y = np.asarray(y)
    bins = pd.qcut(y, q=min(q, len(np.unique(y))), duplicates="drop", labels=False)
    return np.asarray(bins)

Device: cpu


In [ ]:
class MLPRegressor(nn.Module):
    def __init__(self, in_dim, hidden, p_drop, n_layers, norm_type, act_name):
        super().__init__()

        # Define the dimensions of each layer: input + hidden layers
        dims = [in_dim] + [hidden] * n_layers

        # Lists to store fully connected layers and normalization layers
        self.fcs = nn.ModuleList()
        self.norms = nn.ModuleList()
        # Dropout layer for regularization
        self.drop = nn.Dropout(p_drop)
        # Select activation function
        if act_name == "relu":
            self.act = F.relu
        elif act_name == "gelu":
            self.act = F.gelu
        elif act_name == "silu":
            self.act = F.silu
        else:
            raise ValueError(f"Activation '{act_name}' not supported")
        # Build hidden layers
        for i in range(n_layers):
            self.fcs.append(nn.Linear(dims[i], dims[i + 1]))
            # Normalization layer
            if norm_type == "batchnorm":
                self.norms.append(nn.BatchNorm1d(dims[i + 1]))
            elif norm_type == "layernorm":
                self.norms.append(nn.LayerNorm(dims[i + 1]))
            elif norm_type == "none":
                self.norms.append(nn.Identity())
            else:
                raise ValueError(f"Norm '{norm_type}' not supported")
        # Final output layer (regression -> single value)
        self.out = nn.Linear(hidden, 1)  #takes the last hidden layer and generates a single number

    def forward(self, x):
        # Pass input through each hidden layer
        for fc, norm in zip(self.fcs, self.norms):
            x = fc(x)
            x = norm(x)
            x = self.act(x)
            x = self.drop(x)
        # Output layer and squeeze to remove last dimension
        return self.out(x).squeeze(-1)

In [27]:
def train_one_fold_mlp(
    X_tr, y_tr, X_va, y_va,
    hidden, n_layers, dropout,
    lr, weight_decay, batch_size,
    clip_norm, beta_huber, epochs,
    norm_type, act_name,
    patience=20,
    min_delta=1e-4,
    seed=42,
    return_details=False
):
    """
    Train an MLP on a training fold and monitor a separate validation fold
    for early stopping.

    Parameters
    ----------
    X_tr, y_tr : array-like
        Training features and targets.
    X_va, y_va : array-like
        Validation features and targets, used for early stopping
        and validation monitoring inside this training routine.
    return_details : bool, default=False
        If True, return predictions and additional training metadata.

    Returns
    -------
    np.ndarray or dict
        Validation predictions, or a dictionary with predictions and details.
    """
    set_seed(seed)

    Xtr_t = torch.tensor(X_tr, dtype=torch.float32)
    ytr_t = torch.tensor(y_tr, dtype=torch.float32)
    Xva_t = torch.tensor(X_va, dtype=torch.float32)
    yva_t = torch.tensor(y_va, dtype=torch.float32)

    tr_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(Xtr_t, ytr_t),
        batch_size=batch_size,
        shuffle=True,
        drop_last=False
    )

    model = MLPRegressor(
        in_dim=X_tr.shape[1],
        hidden=hidden,
        p_drop=dropout,
        n_layers=n_layers,
        norm_type=norm_type,
        act_name=act_name
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

    best_val_loss = np.inf
    best_epoch = 0
    patience_counter = 0
    best_state = copy.deepcopy(model.state_dict())

    train_loss_history = []
    val_loss_history = []

    for epoch in range(1, epochs + 1):
        model.train()
        batch_losses = []

        for xb, yb in tr_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
                pred = model(xb)
                loss = F.smooth_l1_loss(pred, yb, beta=beta_huber)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), clip_norm)
            scaler.step(optimizer)
            scaler.update()

            batch_losses.append(loss.item())

        mean_train_loss = float(np.mean(batch_losses)) if len(batch_losses) > 0 else np.nan
        train_loss_history.append(mean_train_loss)

        model.eval()
        with torch.no_grad():
            with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
                val_pred = model(Xva_t.to(device))
                val_loss = F.smooth_l1_loss(val_pred, yva_t.to(device), beta=beta_huber).item()

        val_loss_history.append(val_loss)

        if val_loss < (best_val_loss - min_delta):
            best_val_loss = val_loss
            best_epoch = epoch
            patience_counter = 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            patience_counter += 1

        if patience_counter >= patience:
            break

    model.load_state_dict(best_state)
    model.eval()

    with torch.no_grad():
        preds = model(Xva_t.to(device)).cpu().numpy()

    if return_details:
        return {
            "preds": preds,
            "best_epoch": best_epoch,
            "best_val_loss": best_val_loss,
            "train_loss_history": train_loss_history,
            "val_loss_history": val_loss_history,
            "model_state_dict": best_state
        }

    return preds

In [28]:
def train_mlp_fixed_epochs(
    X_tr, y_tr, X_te,
    hidden, n_layers, dropout,
    lr, weight_decay, batch_size,
    clip_norm, beta_huber, epochs,
    norm_type, act_name,
    seed=42
):
    set_seed(seed)

    Xtr_t = torch.tensor(X_tr, dtype=torch.float32)
    ytr_t = torch.tensor(y_tr, dtype=torch.float32)
    Xte_t = torch.tensor(X_te, dtype=torch.float32)

    tr_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(Xtr_t, ytr_t),
        batch_size=batch_size,
        shuffle=True,
        drop_last=False
    )

    model = MLPRegressor(
        in_dim=X_tr.shape[1],
        hidden=hidden,
        p_drop=dropout,
        n_layers=n_layers,
        norm_type=norm_type,
        act_name=act_name
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

    train_loss_history = []

    for epoch in range(1, epochs + 1):
        model.train()
        batch_losses = []

        for xb, yb in tr_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
                pred = model(xb)
                loss = F.smooth_l1_loss(pred, yb, beta=beta_huber)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), clip_norm)
            scaler.step(optimizer)
            scaler.update()

            batch_losses.append(loss.item())

        mean_train_loss = float(np.mean(batch_losses)) if len(batch_losses) > 0 else np.nan
        train_loss_history.append(mean_train_loss)

    model.eval()
    with torch.no_grad():
        preds_te = model(Xte_t.to(device)).cpu().numpy()

    return {
        "preds": preds_te,
        "train_loss_history": train_loss_history
    }

In [29]:
def train_eval_mlp_inner_clean(
    X_tr, y_tr, X_va, y_va,
    hidden, n_layers, dropout,
    lr, weight_decay, batch_size,
    clip_norm, beta_huber, epochs,
    norm_type, act_name,
    patience=20,
    min_delta=1e-4,
    seed=42,
    es_val_size=0.15
):
    """
    Clean inner-loop training for nested CV.

    The inner validation fold (X_va, y_va) is used ONLY for scoring the trial.
    Early stopping is performed on a separate validation split carved out
    exclusively from the inner training fold (X_tr, y_tr).

    Steps
    -----
    1. Split X_tr/y_tr into:
       - sub-training set
       - early-stopping validation set
    2. Train with early stopping on that internal validation split
       to estimate the best epoch.
    3. Retrain on the full inner training fold (X_tr, y_tr)
       for best_epoch epochs.
    4. Predict on the untouched inner validation fold (X_va).
    """
    set_seed(seed)

    es_bins = make_strat_bins(y_tr, q=10)
    idx = np.arange(len(X_tr))

    subtr_idx, es_idx = train_test_split(
        idx,
        test_size=es_val_size,
        random_state=seed,
        stratify=es_bins
    )

    X_subtr, y_subtr = X_tr[subtr_idx], y_tr[subtr_idx]
    X_es, y_es = X_tr[es_idx], y_tr[es_idx]

    fit_info = train_one_fold_mlp(
        X_subtr, y_subtr, X_es, y_es,
        hidden=hidden,
        n_layers=n_layers,
        dropout=dropout,
        lr=lr,
        weight_decay=weight_decay,
        batch_size=batch_size,
        clip_norm=clip_norm,
        beta_huber=beta_huber,
        epochs=epochs,
        norm_type=norm_type,
        act_name=act_name,
        patience=patience,
        min_delta=min_delta,
        seed=seed,
        return_details=True
    )

    best_epoch = max(1, fit_info["best_epoch"])

    final_fit = train_mlp_fixed_epochs(
        X_tr, y_tr, X_va,
        hidden=hidden,
        n_layers=n_layers,
        dropout=dropout,
        lr=lr,
        weight_decay=weight_decay,
        batch_size=batch_size,
        clip_norm=clip_norm,
        beta_huber=beta_huber,
        epochs=best_epoch,
        norm_type=norm_type,
        act_name=act_name,
        seed=seed
    )

    return final_fit["preds"]

In [ ]:
def fit_predict_mlp_outer(
    X_tr, y_tr, X_te,
    hidden, n_layers, dropout,
    lr, weight_decay, batch_size,
    clip_norm, beta_huber, epochs,
    norm_type, act_name,
    patience=20,
    min_delta=1e-4,
    seed=42,
    val_size=0.15
):
    set_seed(seed)

    # Internal split ONLY for estimating best_epoch
    inner_bins_for_es = make_strat_bins(y_tr, q=10)
    idx = np.arange(len(X_tr))

    subtr_idx, val_idx = train_test_split(
        idx,
        test_size=val_size,
        random_state=seed,
        stratify=inner_bins_for_es
    )

    X_subtr, y_subtr = X_tr[subtr_idx], y_tr[subtr_idx]
    X_val, y_val = X_tr[val_idx], y_tr[val_idx]

    # step A: early stopping to found best_epoch
    fit_info = train_one_fold_mlp(
        X_subtr, y_subtr, X_val, y_val,
        hidden=hidden,
        n_layers=n_layers,
        dropout=dropout,
        lr=lr,
        weight_decay=weight_decay,
        batch_size=batch_size,
        clip_norm=clip_norm,
        beta_huber=beta_huber,
        epochs=epochs,
        norm_type=norm_type,
        act_name=act_name,
        patience=patience,
        min_delta=min_delta,
        seed=seed,
        return_details=True
    )

    best_epoch = fit_info["best_epoch"]
    best_val_loss = fit_info["best_val_loss"]

    # security: avoid best_epoch = 0
    if best_epoch < 1:
        best_epoch = 1

    # step B: retrain all X_tr for best_epoch
    final_fit = train_mlp_fixed_epochs(
        X_tr, y_tr, X_te,
        hidden=hidden,
        n_layers=n_layers,
        dropout=dropout,
        lr=lr,
        weight_decay=weight_decay,
        batch_size=batch_size,
        clip_norm=clip_norm,
        beta_huber=beta_huber,
        epochs=best_epoch,
        norm_type=norm_type,
        act_name=act_name,
        seed=seed
    )

    return {
        "preds": final_fit["preds"],
        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,
        "train_loss_history_es": fit_info["train_loss_history"],
        "val_loss_history_es": fit_info["val_loss_history"],
        "train_loss_history_final": final_fit["train_loss_history"]
    }

In [ ]:
def objective(trial, model_name, X_outer_tr, y_outer_tr, inner_splits):
    # Define hyperparameter search space depending on the model
    if model_name == "MLP":
        params = {
            "hidden": trial.suggest_int("hidden", 128, 1024, step=128),
            "n_layers": trial.suggest_int("n_layers", 1, 6),
            "dropout": trial.suggest_float("dropout", 0.0, 0.35),
            "lr": trial.suggest_float("lr", 1e-5, 1e-2, log=True),
            "weight_decay": trial.suggest_float("weight_decay", 1e-8, 1e-2, log=True),
            "batch_size": trial.suggest_categorical("batch_size", [16, 32, 64]),
            "clip_norm": trial.suggest_float("clip_norm", 0.5, 5.0),
            "beta_huber": trial.suggest_float("beta_huber", 0.5, 2.0),
            "epochs": trial.suggest_int("epochs", 100, 300),
            "norm": trial.suggest_categorical("norm", ["batchnorm", "layernorm", "none"]),
            "act": trial.suggest_categorical("act", ["relu", "gelu", "silu"]),
            "patience": trial.suggest_int("patience", 10, 30),
            "min_delta": trial.suggest_float("min_delta", 1e-5, 1e-3, log=True),
        }

    elif model_name == "XGB":
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 500, step=100),
            "max_depth": trial.suggest_int("max_depth", 3, 6),
            "learning_rate": trial.suggest_float("learning_rate", 1e-2, 1e-1, log=True),
            "subsample": trial.suggest_float("subsample", 0.7, 1.0, step=0.1),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0, step=0.1),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 1e-1, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 1.0, log=True),
        }

    elif model_name == "SVR":
        params = {
            "C": trial.suggest_float("C", 1e-1, 1e2, log=True),
            "epsilon": trial.suggest_float("epsilon", 1e-2, 0.5, log=True),
            "gamma": trial.suggest_categorical("gamma", ["scale"]),
            "kernel": trial.suggest_categorical("kernel", ["rbf", "linear"]),
        }

    elif model_name == "RF":
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 500, step=100),
            "max_depth": trial.suggest_int("max_depth", 8, 20, step=4),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 6),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 3),
            "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        }
    else:
        raise ValueError(f"Unsupported model_name: {model_name}")

    # Store validation scores across inner folds
    scores = []

    # Loop over inner CV splits
    for tr_idx, va_idx in inner_splits:
        Xtr, Xva = X_outer_tr[tr_idx], X_outer_tr[va_idx]
        ytr, yva = y_outer_tr[tr_idx], y_outer_tr[va_idx]

        # Train and predict depending on model type
        if model_name == "MLP":
            preds = train_eval_mlp_inner_clean(
                Xtr, ytr, Xva, yva,
                hidden=params["hidden"],
                n_layers=params["n_layers"],
                dropout=params["dropout"],
                lr=params["lr"],
                weight_decay=params["weight_decay"],
                batch_size=params["batch_size"],
                clip_norm=params["clip_norm"],
                beta_huber=params["beta_huber"],
                epochs=params["epochs"],
                norm_type=params["norm"],
                act_name=params["act"],
                patience=params["patience"],
                min_delta=params["min_delta"],
                seed=RANDOM_STATE,
                es_val_size=0.15
            )

        elif model_name == "XGB":
            # Gradient boosting with decision trees
            model = xgb.XGBRegressor(
                n_estimators=params["n_estimators"],
                max_depth=params["max_depth"],
                learning_rate=params["learning_rate"],
                subsample=params["subsample"],
                colsample_bytree=params["colsample_bytree"],
                reg_alpha=params["reg_alpha"],
                reg_lambda=params["reg_lambda"],
                objective="reg:squarederror",
                random_state=RANDOM_STATE,
                n_jobs=-1
            )
            preds = model.fit(Xtr, ytr).predict(Xva)

        elif model_name == "SVR":
            # Pipeline
            model = Pipeline([
                ("svr", SVR(
                    C=params["C"],
                    epsilon=params["epsilon"],
                    gamma=params["gamma"],
                    kernel=params["kernel"]
                ))
            ])
            preds = model.fit(Xtr, ytr).predict(Xva)

        elif model_name == "RF":
            # Random Forest ensemble
            model = RandomForestRegressor(
                n_estimators=params["n_estimators"],
                max_depth=params["max_depth"],
                min_samples_split=params["min_samples_split"],
                min_samples_leaf=params["min_samples_leaf"],
                max_features=params["max_features"],
                random_state=RANDOM_STATE,
                n_jobs=-1
            )
            preds = model.fit(Xtr, ytr).predict(Xva)


        # Compute RMSE on validation fold
        scores.append(rmse(yva, preds))

    # Return mean validation error across inner folds
    return float(np.mean(scores))

In [ ]:
MODEL_NAMES = ["MLP", "XGB", "SVR", "RF"]

outer_bins = make_strat_bins(y_all, q=10)

outer_skf = StratifiedKFold(
    n_splits=OUTER_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

results = []
best_params_by_fold = []
outer_predictions = []

outer_iterator = list(outer_skf.split(X_all, outer_bins))

for model_name in tqdm(MODEL_NAMES, desc="Models"):
    print(f"\nMODEL: {model_name}")

    for fold, (train_idx, test_idx) in enumerate(
        tqdm(outer_iterator, desc=f"{model_name} - Outer CV", leave=False),
        start=1
    ):
        X_tr, X_te = X_all[train_idx], X_all[test_idx]
        y_tr, y_te = y_all[train_idx], y_all[test_idx]

        inner_bins = make_strat_bins(y_tr, q=10)

        inner_skf = StratifiedKFold(
            n_splits=INNER_SPLITS,
            shuffle=True,
            random_state=RANDOM_STATE
        )

        inner_splits = list(inner_skf.split(X_tr, inner_bins))

        study = optuna.create_study(direction="minimize")
        study.optimize(
            lambda t: objective(t, model_name, X_tr, y_tr, inner_splits),
            n_trials=N_TRIALS,
            show_progress_bar=True
        )

        best_params = study.best_params
        best_score = study.best_value

        print(f"Fold {fold} best params: {best_params}")
        print(f"Fold {fold} best inner RMSE: {best_score:.4f}")

        # final training on outer train / final eval on outer test
        if model_name == "MLP":
            mlp_out = fit_predict_mlp_outer(
                X_tr, y_tr, X_te,
                hidden=best_params["hidden"],
                n_layers=best_params["n_layers"],
                dropout=best_params["dropout"],
                lr=best_params["lr"],
                weight_decay=best_params["weight_decay"],
                batch_size=best_params["batch_size"],
                clip_norm=best_params["clip_norm"],
                beta_huber=best_params["beta_huber"],
                epochs=best_params["epochs"],
                norm_type=best_params["norm"],
                act_name=best_params["act"],
                patience=best_params["patience"],
                min_delta=best_params["min_delta"],
                seed=RANDOM_STATE
            )
            preds = mlp_out["preds"]
            best_epoch = mlp_out["best_epoch"]
            best_val_loss = mlp_out["best_val_loss"]

        elif model_name == "XGB":
            model = xgb.XGBRegressor(
                n_estimators=best_params["n_estimators"],
                max_depth=best_params["max_depth"],
                learning_rate=best_params["learning_rate"],
                subsample=best_params["subsample"],
                colsample_bytree=best_params["colsample_bytree"],
                reg_alpha=best_params["reg_alpha"],
                reg_lambda=best_params["reg_lambda"],
                objective="reg:squarederror",
                random_state=RANDOM_STATE,
                n_jobs=-1
            )
            preds = model.fit(X_tr, y_tr).predict(X_te)

        elif model_name == "SVR":
            model = Pipeline([
                ("svr", SVR(
                    C=best_params["C"],
                    epsilon=best_params["epsilon"],
                    gamma=best_params["gamma"],
                    kernel=best_params["kernel"]
                ))
            ])
            preds = model.fit(X_tr, y_tr).predict(X_te)

        elif model_name == "RF":
            model = RandomForestRegressor(
                n_estimators=best_params["n_estimators"],
                max_depth=best_params["max_depth"],
                min_samples_split=best_params["min_samples_split"],
                min_samples_leaf=best_params["min_samples_leaf"],
                max_features=best_params["max_features"],
                random_state=RANDOM_STATE,
                n_jobs=-1
            )
            preds = model.fit(X_tr, y_tr).predict(X_te)

        else:
            raise ValueError(f"Unsupported model_name: {model_name}")

        metrics = compute_metrics(y_te, preds)

        row = {
            "model": model_name,
            "fold": fold,
            "rmse": metrics["rmse"],
            "r2": metrics["r2"],
            "pearson": metrics["pearson"],
            "best_inner_rmse": best_score,
            **best_params
        }

        if model_name == "MLP":
            row["best_epoch"] = best_epoch
            row["best_val_loss"] = best_val_loss

        results.append(row)

        best_params_by_fold.append({
            "model": model_name,
            "fold": fold,
            "best_inner_rmse": best_score,
            **best_params
        })

        # save outer fold predictions 
        for idx_sample, y_true_i, y_pred_i in zip(test_idx, y_te, preds):
            pred_row = {
                "model": model_name,
                "fold": fold,
                "sample_idx": int(idx_sample),
                "y_true": float(y_true_i),
                "y_pred": float(y_pred_i)
            }

            if model_name == "MLP":
                pred_row["best_epoch"] = best_epoch

            outer_predictions.append(pred_row)

        print(f"Fold {fold} outer metrics: RMSE={metrics['rmse']:.4f} | R2={metrics['r2']:.4f} | Pearson={metrics['pearson']:.4f}")

    # save partial results after each model
    results_model_df = pd.DataFrame([r for r in results if r["model"] == model_name])
    best_params_model_df = pd.DataFrame([r for r in best_params_by_fold if r["model"] == model_name])
    outer_preds_model_df = pd.DataFrame([r for r in outer_predictions if r["model"] == model_name])

    results_model_df.to_csv(f"results_{model_name}.csv", index=False)
    best_params_model_df.to_csv(f"best_params_{model_name}.csv", index=False)
    outer_preds_model_df.to_csv(f"outer_predictions_{model_name}.csv", index=False)

    print(f"\nSaved partial files for {model_name}:")
    print(f"- results_{model_name}.csv")
    print(f"- best_params_{model_name}.csv")
    print(f"- outer_predictions_{model_name}.csv")

#final save all together
results_df = pd.DataFrame(results)
best_params_df = pd.DataFrame(best_params_by_fold)
outer_predictions_df = pd.DataFrame(outer_predictions)

results_df.to_csv("results_all_models.csv", index=False)
best_params_df.to_csv("best_params_all_models.csv", index=False)
outer_predictions_df.to_csv("outer_predictions_all_models.csv", index=False)

print("\nSaved final files:")
print("- results_all_models.csv")
print("- best_params_all_models.csv")
print("- outer_predictions_all_models.csv")


Models:   0%|          | 0/4 [00:00<?, ?it/s]


MODEL: MLP


  0%|          | 0/20 [00:00<?, ?it/s]

Fold 1 best params: {'hidden': 384, 'n_layers': 1, 'dropout': 0.22364846409658748, 'lr': 0.009653777344350991, 'weight_decay': 0.009463643440176327, 'batch_size': 32, 'clip_norm': 1.9548140002353454, 'beta_huber': 1.528346088314889, 'epochs': 119, 'norm': 'none', 'act': 'relu', 'patience': 15, 'min_delta': 0.00020419445742895628}
Fold 1 best inner RMSE: 0.8019


Fold 1 outer metrics: RMSE=0.8907 | R2=0.2335 | Pearson=0.5509


...

Fold 9 outer metrics: RMSE=0.8648 | R2=0.3404 | Pearson=0.5854


  0%|          | 0/20 [00:00<?, ?it/s]

Fold 10 best params: {'n_estimators': 100, 'max_depth': 20, 'min_samples_split': 3, 'min_samples_leaf': 1, 'max_features': 'sqrt'}
Fold 10 best inner RMSE: 0.7588


Models: 100%|██████████| 4/4 [14:04:23<00:00, 12665.90s/it]  

Fold 10 outer metrics: RMSE=0.7473 | R2=0.4157 | Pearson=0.6675

Saved partial files for RF:
- results_RF.csv
- best_params_RF.csv
- outer_predictions_RF.csv

Saved final files:
- results_all_models.csv
- best_params_all_models.csv
- outer_predictions_all_models.csv


In [33]:
results_df = pd.DataFrame(results)
best_params_df = pd.DataFrame(best_params_by_fold)

summary = results_df.groupby("model").agg(
    mean_r2=("r2", "mean"),
    std_r2=("r2", "std"),
    mean_rmse=("rmse", "mean"),
    std_rmse=("rmse", "std"),
    mean_pearson=("pearson", "mean"),
    std_pearson=("pearson", "std"),
).reset_index()

print(results_df.shape)
print(best_params_df.shape)
print(summary)

results_df.to_csv("nested_cv_results.csv", index=False)
best_params_df.to_csv("nested_cv_best_params.csv", index=False)
summary.to_csv("nested_cv_summary.csv", index=False)

(40, 35)
(40, 30)
  model   mean_r2    std_r2  mean_rmse  std_rmse  mean_pearson  std_pearson
0   MLP  0.440030  0.146117   0.757980  0.112581      0.688060     0.095324
1    RF  0.465483  0.074470   0.744370  0.064482      0.703639     0.070829
2   SVR  0.424876  0.134578   0.769406  0.104012      0.660108     0.092055
3   XGB  0.509955  0.102604   0.711315  0.087120      0.722326     0.078867


### 6. Final Hyperparameter Optimization of the Best-Performing XGBoost Model

After identifying XGBoost as the best-performing model during the nested cross-validation comparison, an additional hyperparameter optimization step was carried out on the full training set to refine its configuration before final model training.

A stratified 10-fold cross-validation procedure was defined by binning the continuous target variable into quantile-based groups, ensuring a balanced target distribution across folds. Optuna was then used to optimize the XGBoost hyperparameters by minimizing the mean validation RMSE across these folds. The search space included tree complexity, learning rate, subsampling, feature subsampling, regularization terms, and split-control parameters.

Once the best hyperparameter configuration was identified, the model was re-evaluated using the same 10-fold cross-validation scheme. For each fold, the model was trained on the training portion and predictions were generated for both the training and validation subsets. Performance metrics, including RMSE, \(R^2\), and Pearson correlation, were computed for each fold in order to assess both fitting behavior and generalization performance.

The predictions and fold-wise metrics were saved to disk for downstream analysis. Finally, all validation and training predictions were aggregated across folds, enabling the construction of global performance plots and a more comprehensive assessment of the optimized XGBoost model.

In [ ]:
# helpers
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def pearson_corr(a, b):
    a = np.asarray(a).reshape(-1)
    b = np.asarray(b).reshape(-1)
    if len(a) < 2:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])

def compute_metrics(y_true, y_pred):
    return {
        "rmse": rmse(y_true, y_pred),
        "r2": r2_score(y_true, y_pred),
        "pearson": pearson_corr(y_true, y_pred)
    }

def make_strat_bins(y, q=10):
    y = np.asarray(y)
    bins = pd.qcut(y, q=min(q, len(np.unique(y))), duplicates="drop", labels=False)
    return np.asarray(bins)

# settings
RANDOM_STATE = 42
FINAL_CV_SPLITS = 10
FINAL_N_TRIALS = 300

final_bins = make_strat_bins(y_all, q=10)

final_skf = StratifiedKFold(
    n_splits=FINAL_CV_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

final_splits = list(final_skf.split(X_all, final_bins))

# Optuna objective for final XGB tuning
def objective_xgb_final(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 800, step=50),
        "max_depth": trial.suggest_int("max_depth", 1, 5),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 1e-2, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 0.9, step=0.1),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 0.9, step=0.1),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 1.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.5, 10.0, log=True),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0.0, 3.0)
    }

    fold_scores = []

    for tr_idx, va_idx in final_splits:
        X_tr, X_va = X_all[tr_idx], X_all[va_idx]
        y_tr, y_va = y_all[tr_idx], y_all[va_idx]

        model = xgb.XGBRegressor(
            n_estimators=params["n_estimators"],
            max_depth=params["max_depth"],
            learning_rate=params["learning_rate"],
            subsample=params["subsample"],
            colsample_bytree=params["colsample_bytree"],
            reg_alpha=params["reg_alpha"],
            reg_lambda=params["reg_lambda"],
            min_child_weight=params["min_child_weight"],
            gamma=params["gamma"],
            objective="reg:squarederror",
            random_state=RANDOM_STATE,
            n_jobs=-1
        )

        preds = model.fit(X_tr, y_tr).predict(X_va)
        fold_scores.append(rmse(y_va, preds))

    return float(np.mean(fold_scores))

# final tuning
study_final_xgb = optuna.create_study(direction="minimize")
study_final_xgb.optimize(
    objective_xgb_final,
    n_trials=FINAL_N_TRIALS,
    show_progress_bar=True
)

best_xgb_params = study_final_xgb.best_params
best_xgb_cv_rmse = study_final_xgb.best_value

print("Best XGB params:", best_xgb_params)
print(f"Best mean CV RMSE: {best_xgb_cv_rmse:.4f}")

# 10-fold CV with best params
train_pred_rows = []
val_pred_rows = []
cv_fold_metrics = []

for fold, (tr_idx, va_idx) in enumerate(final_splits, start=1):
    X_tr, X_va = X_all[tr_idx], X_all[va_idx]
    y_tr, y_va = y_all[tr_idx], y_all[va_idx]

    model = xgb.XGBRegressor(
        n_estimators=best_xgb_params["n_estimators"],
        max_depth=best_xgb_params["max_depth"],
        learning_rate=best_xgb_params["learning_rate"],
        subsample=best_xgb_params["subsample"],
        colsample_bytree=best_xgb_params["colsample_bytree"],
        reg_alpha=best_xgb_params["reg_alpha"],
        reg_lambda=best_xgb_params["reg_lambda"],
        min_child_weight=best_xgb_params["min_child_weight"],
        gamma=best_xgb_params["gamma"],
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    model.fit(X_tr, y_tr)

    preds_tr = model.predict(X_tr)
    preds_va = model.predict(X_va)

    train_metrics = compute_metrics(y_tr, preds_tr)
    val_metrics = compute_metrics(y_va, preds_va)

    cv_fold_metrics.append({
        "fold": fold,
        "train_rmse": train_metrics["rmse"],
        "train_r2": train_metrics["r2"],
        "train_pearson": train_metrics["pearson"],
        "val_rmse": val_metrics["rmse"],
        "val_r2": val_metrics["r2"],
        "val_pearson": val_metrics["pearson"]
    })

    for idx_sample, y_true_i, y_pred_i in zip(tr_idx, y_tr, preds_tr):
        train_pred_rows.append({
            "fold": fold,
            "sample_idx": int(idx_sample),
            "y_true": float(y_true_i),
            "y_pred": float(y_pred_i)
        })

    for idx_sample, y_true_i, y_pred_i in zip(va_idx, y_va, preds_va):
        val_pred_rows.append({
            "fold": fold,
            "sample_idx": int(idx_sample),
            "y_true": float(y_true_i),
            "y_pred": float(y_pred_i)
        })

    print(
        f"Fold {fold} | "
        f"Train RMSE={train_metrics['rmse']:.4f}, R2={train_metrics['r2']:.4f}, Pearson={train_metrics['pearson']:.4f} | "
        f"Val RMSE={val_metrics['rmse']:.4f}, R2={val_metrics['r2']:.4f}, Pearson={val_metrics['pearson']:.4f}"
    )

cv_metrics_df = pd.DataFrame(cv_fold_metrics)
train_pred_df = pd.DataFrame(train_pred_rows)
val_pred_df = pd.DataFrame(val_pred_rows)

print("\nCV summary:")
print(cv_metrics_df.agg({
    "train_rmse": ["mean", "std"],
    "train_r2": ["mean", "std"],
    "train_pearson": ["mean", "std"],
    "val_rmse": ["mean", "std"],
    "val_r2": ["mean", "std"],
    "val_pearson": ["mean", "std"]
}))

cv_metrics_df.to_csv("xgb_final_10fold_cv_metrics.csv", index=False)
train_pred_df.to_csv("xgb_final_10fold_cv_train_predictions.csv", index=False)
val_pred_df.to_csv("xgb_final_10fold_cv_val_predictions.csv", index=False)

print("\nSaved files:")
print("- xgb_final_10fold_cv_metrics.csv")
print("- xgb_final_10fold_cv_train_predictions.csv")
print("- xgb_final_10fold_cv_val_predictions.csv")

fig_path = "figures"
os.makedirs(fig_path, exist_ok=True)

all_preds_val   = np.array([row["y_pred"] for row in val_pred_rows], dtype=float)
all_trues_val   = np.array([row["y_true"] for row in val_pred_rows], dtype=float)
all_preds_train = np.array([row["y_pred"] for row in train_pred_rows], dtype=float)
all_trues_train = np.array([row["y_true"] for row in train_pred_rows], dtype=float)

  0%|          | 0/300 [00:00<?, ?it/s]

Best XGB params: {'n_estimators': 800, 'max_depth': 4, 'learning_rate': 0.008540383060389328, 'subsample': 0.8, 'colsample_bytree': 0.7, 'reg_alpha': 0.022099239841633523, 'reg_lambda': 0.8222837721469007, 'min_child_weight': 6, 'gamma': 0.00021183739862356155}
Best mean CV RMSE: 0.7117
Fold 1 | Train RMSE=0.2546, R2=0.9380, Pearson=0.9788 | Val RMSE=0.7632, R2=0.4373, Pearson=0.6628
Fold 2 | Train RMSE=0.2579, R2=0.9368, Pearson=0.9789 | Val RMSE=0.7418, R2=0.4326, Pearson=0.6618
Fold 3 | Train RMSE=0.2617, R2=0.9348, Pearson=0.9781 | Val RMSE=0.5908, R2=0.6479, Pearson=0.8445
Fold 4 | Train RMSE=0.2480, R2=0.9398, Pearson=0.9797 | Val RMSE=0.7976, R2=0.4903, Pearson=0.7080
Fold 5 | Train RMSE=0.2694, R2=0.9310, Pearson=0.9769 | Val RMSE=0.6127, R2=0.6126, Pearson=0.8070
Fold 6 | Train RMSE=0.2524, R2=0.9395, Pearson=0.9798 | Val RMSE=0.7088, R2=0.4803, Pearson=0.6987
Fold 7 | Train RMSE=0.2609, R2=0.9338, Pearson=0.9773 | Val RMSE=0.7054, R2=0.5778, Pearson=0.8192
Fold 8 | Train RMSE

### Re-running the Final Cross-Validation with Fixed Best Parameters

The following cell was added to rerun only the **final 10-fold cross-validation** using the **best XGBoost hyperparameters previously found by Optuna**.
This avoids launching the full Optuna optimization again, which is computationally expensive and time-consuming. Instead, the best parameter values are manually specified and used directly to recompute the final cross-validation metrics, training predictions, validation predictions, and output files.
Therefore, this cell is useful when the Optuna search has already been completed and we only need to regenerate or verify the final CV results with the selected best model configuration.


In [ ]:
# helpers
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def pearson_corr(a, b):
    a = np.asarray(a).reshape(-1)
    b = np.asarray(b).reshape(-1)
    if len(a) < 2:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])

def compute_metrics(y_true, y_pred):
    return {
        "rmse": rmse(y_true, y_pred),
        "r2": r2_score(y_true, y_pred),
        "pearson": pearson_corr(y_true, y_pred)
    }

def make_strat_bins(y, q=10):
    y = np.asarray(y)
    bins = pd.qcut(y, q=min(q, len(np.unique(y))), duplicates="drop", labels=False)
    return np.asarray(bins)

# settings
RANDOM_STATE = 42
FINAL_CV_SPLITS = 10

# SET MANUAL BEST PARAMS

best_xgb_params = {
    "n_estimators": 800,
    "max_depth": 4,
    "learning_rate": 0.008540383060389328,
    "subsample": 0.8,
    "colsample_bytree": 0.7,
    "reg_alpha": 0.022099239841633523,
    "reg_lambda": 0.8222837721469007,
    "min_child_weight": 6,
    "gamma": 0.00021183739862356155
}

print("Manual XGB params:", best_xgb_params)

# stratified folds
final_bins = make_strat_bins(y_all, q=10)

final_skf = StratifiedKFold(
    n_splits=FINAL_CV_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

final_splits = list(final_skf.split(X_all, final_bins))

# 10-fold CV with manually selected params
train_pred_rows = []
val_pred_rows = []
cv_fold_metrics = []

for fold, (tr_idx, va_idx) in enumerate(final_splits, start=1):
    X_tr, X_va = X_all[tr_idx], X_all[va_idx]
    y_tr, y_va = y_all[tr_idx], y_all[va_idx]

    model = xgb.XGBRegressor(
        n_estimators=best_xgb_params["n_estimators"],
        max_depth=best_xgb_params["max_depth"],
        learning_rate=best_xgb_params["learning_rate"],
        subsample=best_xgb_params["subsample"],
        colsample_bytree=best_xgb_params["colsample_bytree"],
        reg_alpha=best_xgb_params["reg_alpha"],
        reg_lambda=best_xgb_params["reg_lambda"],
        min_child_weight=best_xgb_params["min_child_weight"],
        gamma=best_xgb_params["gamma"],
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    model.fit(X_tr, y_tr)

    preds_tr = model.predict(X_tr)
    preds_va = model.predict(X_va)

    train_metrics = compute_metrics(y_tr, preds_tr)
    val_metrics = compute_metrics(y_va, preds_va)

    cv_fold_metrics.append({
        "fold": fold,
        "train_rmse": train_metrics["rmse"],
        "train_r2": train_metrics["r2"],
        "train_pearson": train_metrics["pearson"],
        "val_rmse": val_metrics["rmse"],
        "val_r2": val_metrics["r2"],
        "val_pearson": val_metrics["pearson"]
    })

    for idx_sample, y_true_i, y_pred_i in zip(tr_idx, y_tr, preds_tr):
        train_pred_rows.append({
            "fold": fold,
            "sample_idx": int(idx_sample),
            "y_true": float(y_true_i),
            "y_pred": float(y_pred_i)
        })

    for idx_sample, y_true_i, y_pred_i in zip(va_idx, y_va, preds_va):
        val_pred_rows.append({
            "fold": fold,
            "sample_idx": int(idx_sample),
            "y_true": float(y_true_i),
            "y_pred": float(y_pred_i)
        })

    print(
        f"Fold {fold} | "
        f"Train RMSE={train_metrics['rmse']:.4f}, "
        f"R2={train_metrics['r2']:.4f}, "
        f"Pearson={train_metrics['pearson']:.4f} | "
        f"Val RMSE={val_metrics['rmse']:.4f}, "
        f"R2={val_metrics['r2']:.4f}, "
        f"Pearson={val_metrics['pearson']:.4f}"
    )

cv_metrics_df = pd.DataFrame(cv_fold_metrics)
train_pred_df = pd.DataFrame(train_pred_rows)
val_pred_df = pd.DataFrame(val_pred_rows)

print("\nCV summary:")
print(cv_metrics_df.agg({
    "train_rmse": ["mean", "std"],
    "train_r2": ["mean", "std"],
    "train_pearson": ["mean", "std"],
    "val_rmse": ["mean", "std"],
    "val_r2": ["mean", "std"],
    "val_pearson": ["mean", "std"]
}))

cv_metrics_df.to_csv("xgb_manual_10fold_cv_metrics.csv", index=False)
train_pred_df.to_csv("xgb_manual_10fold_cv_train_predictions.csv", index=False)
val_pred_df.to_csv("xgb_manual_10fold_cv_val_predictions.csv", index=False)

print("\nSaved files:")
print("- xgb_manual_10fold_cv_metrics.csv")
print("- xgb_manual_10fold_cv_train_predictions.csv")
print("- xgb_manual_10fold_cv_val_predictions.csv")

all_preds_val   = np.array([row["y_pred"] for row in val_pred_rows], dtype=float)
all_trues_val   = np.array([row["y_true"] for row in val_pred_rows], dtype=float)
all_preds_train = np.array([row["y_pred"] for row in train_pred_rows], dtype=float)
all_trues_train = np.array([row["y_true"] for row in train_pred_rows], dtype=float)

Manual XGB params: {'n_estimators': 800, 'max_depth': 4, 'learning_rate': 0.008540383060389328, 'subsample': 0.8, 'colsample_bytree': 0.7, 'reg_alpha': 0.022099239841633523, 'reg_lambda': 0.8222837721469007, 'min_child_weight': 6, 'gamma': 0.00021183739862356155}
Fold 1 | Train RMSE=0.2515, R2=0.9394, Pearson=0.9793 | Val RMSE=0.7709, R2=0.4258, Pearson=0.6543
Fold 2 | Train RMSE=0.2536, R2=0.9389, Pearson=0.9796 | Val RMSE=0.7345, R2=0.4437, Pearson=0.6705
Fold 3 | Train RMSE=0.2627, R2=0.9342, Pearson=0.9781 | Val RMSE=0.5730, R2=0.6688, Pearson=0.8579
Fold 4 | Train RMSE=0.2474, R2=0.9401, Pearson=0.9797 | Val RMSE=0.8015, R2=0.4853, Pearson=0.7050
Fold 5 | Train RMSE=0.2659, R2=0.9328, Pearson=0.9772 | Val RMSE=0.6133, R2=0.6119, Pearson=0.8078
Fold 6 | Train RMSE=0.2558, R2=0.9378, Pearson=0.9792 | Val RMSE=0.7088, R2=0.4804, Pearson=0.6982
Fold 7 | Train RMSE=0.2612, R2=0.9337, Pearson=0.9771 | Val RMSE=0.7126, R2=0.5691, Pearson=0.8071
Fold 8 | Train RMSE=0.2529, R2=0.9391, Pear

## 7. Load Test Set

In this cell we load the **final test dataset**, which will be used **exclusively for the final evaluation of the model**.

The test set must **not be used during**:

- model training
- hyperparameter tuning
- model selection

Therefore, it provides an **unbiased estimate of the model performance** on completely unseen data.

We also verify that the dataset contains the required columns:

- `protbert_sequence` → protein sequence used to compute **ProtBERT embeddings**
- `canonical_smiles` → ligand SMILES used to compute **ChemBERTa embeddings**

In [13]:
#load test set.csv
TEST_CSV_PATH = "test_set.csv" 
df_test = pd.read_csv(TEST_CSV_PATH)
# expeted columns
REQ_COLS = ["protbert_sequence", "canonical_smiles"]
for c in REQ_COLS:
    if c not in df_test.columns:
        raise KeyError(f"Column ‘{c}’ is missing from the test. Available columns: {list(df_test.columns)}")
print("df_test shape:", df_test.shape)
print(df_test[REQ_COLS].head())

df_test shape: (178, 9)
                                   protbert_sequence  \
0  E V T E E D I A E I V S R W T G I P V S K L L ...   
1  P Q I T L W Q R P L V T I K I G G Q L K E A L ...   
2  A T Q G V F T L P A N T R F G V T A F A N S S ...   
3  E E E E V E T F A F Q A E I A Q L M S L I I N ...   
4  W G Y G Q D D G P S H W H K L Y P I A Q G D R ...   

                                    canonical_smiles  
0  CNc1ccccc1C(=O)O[C@H]1C[C@H](n2cnc3c(N)ncnc32)...  
1  O=C(N[C@H]1c2ccccc2C[C@H]1O)[C@H](OCc1ccccc1)[...  
2        CO[C@H]1O[C@H](CO)[C@@H](O)[C@H](O)[C@@H]1O  
3    CCCCCCN(C)C(=O)c1cc(-c2ccnn2-c2ccccc2C)c(O)cc1O  
4             NS(=O)(=O)c1c(F)c(F)c(N2CCCCC2)c(F)c1F  


## Exact Train–Test Overlap Check and Test Set Cleaning

This cell checks for **exact overlap between the training and test sets** at the **ligand–protein pair level**, in order to detect and remove potential data leakage.

### Ligand and Protein Normalization
To make the comparison robust:
- ligand structures are converted from **SMILES** to **InChIKey**, providing a standardized identifier for the same chemical compound;
- protein sequences are **normalized** by converting them to uppercase and removing spaces, separators, and non-alphabetic characters.

### Exact Pair Matching
After normalization, each sample is represented by a unique pair:

- **ligand InChIKey**
- **normalized protein sequence**

The code then identifies all pairs that appear **both in the training set and in the test set**.

### Overlap Inspection
If overlaps are found, the code:
- counts the number of overlapping ligand–protein pairs,
- retrieves the corresponding original rows from both datasets,
- prints the indices of the overlapping samples,

### Test Set Cleaning
Any test rows corresponding to exact ligand–protein overlaps are removed from the test set.  
The cleaned test set is then overwritten.

This step is important to ensure that the test set remains **independent from the training set**, preventing inflated performance estimates caused by exact duplicate samples.

In [14]:
# EXACT overlap check between TRAIN and TEST
# Same ligand (InChIKey) + same protein (normalized sequence)

print("Train shape:", df_train.shape)
print("Test shape :", df_test.shape)
SMILES_COL = "canonical_smiles"
PROTEIN_COL = "protein_sequence"

def normalize_sequence(seq):
    if pd.isna(seq):
        return None
    seq = str(seq).upper().strip()
    seq = seq.replace(" ", "").replace("|", "")
    seq = re.sub(r"[^A-Z]", "", seq)
    return seq if seq else None

def smiles_to_inchikey(smiles):
    if pd.isna(smiles):
        return None
    smiles = str(smiles).strip()
    if smiles == "":
        return None
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        return inchi.MolToInchiKey(mol)
    except Exception:
        return None
    
# copy of dataframes
train_overlap = df_train.copy()
test_overlap = df_test.copy()
# Save the original test index
test_overlap["original_test_index"] = test_overlap.index
train_overlap["original_train_index"] = train_overlap.index
# Normalization
train_overlap["protein_norm"] = train_overlap[PROTEIN_COL].apply(normalize_sequence)
test_overlap["protein_norm"] = test_overlap[PROTEIN_COL].apply(normalize_sequence)
train_overlap["inchikey"] = train_overlap[SMILES_COL].apply(smiles_to_inchikey)
test_overlap["inchikey"] = test_overlap[SMILES_COL].apply(smiles_to_inchikey)
# We only keep valid lines
train_valid = train_overlap.dropna(subset=["protein_norm", "inchikey"]).copy()
test_valid = test_overlap.dropna(subset=["protein_norm", "inchikey"]).copy()

print("Valid train rows:", len(train_valid))
print("Valid test rows :", len(test_valid))
# Unique keys: same ligand + same protein
train_keys = train_valid[["inchikey", "protein_norm"]].drop_duplicates().copy()
test_keys = test_valid[["inchikey", "protein_norm"]].drop_duplicates().copy()
# Intersection between train and test
overlap_keys = pd.merge(
    test_keys,
    train_keys,
    on=["inchikey", "protein_norm"],
    how="inner"
).drop_duplicates()

print("\n=== EXACT ligand + protein overlap between TRAIN and TEST ===")
print("Unique train pairs (ligand, protein):", len(train_keys))
print("Unique test pairs  (ligand, protein):", len(test_keys))
print("Overlapping unique pairs            :", len(overlap_keys))
#  Retrieve the original lines of the test that match
test_exact_overlaps = pd.merge(
    test_valid,
    overlap_keys,
    on=["inchikey", "protein_norm"],
    how="inner"
)

train_exact_overlaps = pd.merge(
    train_valid,
    overlap_keys,
    on=["inchikey", "protein_norm"],
    how="inner"
)

# HERE take the original corrected indexes
test_overlap_indices = sorted(test_exact_overlaps["original_test_index"].unique().tolist())
train_overlap_indices = sorted(train_exact_overlaps["original_train_index"].unique().tolist())

print("Row index in TEST with overlap:", test_overlap_indices)
print("Row index in TRAIN with overlap:", train_overlap_indices)

print("Test rows involved in overlap       :", len(test_exact_overlaps))
print("Train rows involved in overlap      :", len(train_exact_overlaps))

if len(overlap_keys) == 0:
    print("No samples with the same ligand and the same protein in the training and test sets.")

# Remove overlapping rows from TEST and save cleaned test set
df_test = df_test.drop(index=test_overlap_indices)
df_test.to_csv("test_set.csv", index=False)


Train shape: (709, 11)
Test shape : (178, 9)


[15:58:57] WARNING: not removing hydrogen atom without neighbors
[15:58:57] WARNING: not removing hydrogen atom without neighbors
[15:58:57] WARNING: not removing hydrogen atom without neighbors


Valid train rows: 709
Valid test rows : 178

=== EXACT ligand + protein overlap between TRAIN and TEST ===
Unique train pairs (ligand, protein): 709
Unique test pairs  (ligand, protein): 178
Overlapping unique pairs            : 1
Row index in TEST with overlap: [137]
Row index in TRAIN with overlap: [101]
Test rows involved in overlap       : 1
Train rows involved in overlap      : 1


## Compute ProtBERT Embeddings for the Test Set

In this step we compute **protein embeddings using ProtBERT**.

The pipeline performs the following operations:

1. Protein sequence normalization
2. Tokenization in the format expected by ProtBERT
3. Automatic derivation of the maximum allowed length from `max_position_embeddings`
4. Full-length inference without chunking, since test sequences are within the model limit
5. Extraction of latent representations from the transformer model
6. Mean pooling across residue tokens (excluding CLS and SEP)
7. L2 normalization of the resulting embedding vectors

Each protein sequence is therefore represented by a **1024-dimensional embedding vector**.

The embeddings are saved to disk so they do not need to be recomputed.


In [ ]:
# Model parameters
PROT_MODEL_NAME = "Rostlab/prot_bert"  # hidden size = 1024
PROT_DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
PROT_MAX_TOKENS = None
PROT_SEQ_COL    = "protbert_sequence"
EPS = 1e-12
# Reuse ProtBERT helper functions defined in the training embedding cell.
required_helpers = ["normalize_protbert_input", "embed_protein"]
missing_helpers = [name for name in required_helpers if name not in globals()]
if missing_helpers:
    raise NameError(
        "Run the ProtBERT training embedding cell first; missing helpers: "
        + ", ".join(missing_helpers)
    )

# load tokenizer/model
if "prot_tokenizer" not in globals() or "prot_model" not in globals():
    print(f"Load ProtBERT: {PROT_MODEL_NAME} (device={PROT_DEVICE})")
    prot_tokenizer = BertTokenizer.from_pretrained(PROT_MODEL_NAME, do_lower_case=False)
    prot_model = BertModel.from_pretrained(PROT_MODEL_NAME).to(PROT_DEVICE).eval()

PROT_MAX_TOKENS = (
    prot_model.config.max_position_embeddings
    - prot_tokenizer.num_special_tokens_to_add(pair=False)
)
print(
    f"ProtBERT max residue tokens: {PROT_MAX_TOKENS} "
    f"(max_position_embeddings={prot_model.config.max_position_embeddings})"
)

seqs = df_test[PROT_SEQ_COL].tolist()
embs = []

for seq in tqdm(seqs, desc="ProtBERT embeddings (test)", total=len(seqs)):
    spaced = normalize_protbert_input(seq)
    embs.append(
        embed_protein(
            spaced,
            prot_tokenizer,
            prot_model,
            PROT_DEVICE,
            PROT_MAX_TOKENS
        )
    )

test_prot = np.stack(embs, axis=0).astype(np.float32)

# L2 normalize
norms = np.linalg.norm(test_prot, axis=1, keepdims=True)
mask = norms > 0
test_prot = np.where(mask, test_prot / (norms + EPS), test_prot)

print(
    "Test ProtBERT shape:",
    test_prot.shape,
    "mean norm:",
    float(np.mean(np.linalg.norm(test_prot, axis=1)))
)

df_test["prot_embeddings"] = test_prot.tolist()

np.save("test_prot_embeddings.npy", test_prot)
df_test.to_parquet("test_with_prot_embeddings.parquet", index=False, engine="pyarrow")
print("Saved: test_prot_embeddings.npy, test_with_prot_embeddings.parquet")

Carico ProtBERT: Rostlab/prot_bert (device=cpu)
ProtBERT max residue tokens: 39998 (max_position_embeddings=40000)


ProtBERT embeddings (test):   0%|          | 0/177 [00:00<?, ?it/s]

Test ProtBERT shape: (177, 1024) mean norm: 1.0
Saved: test_prot_embeddings.npy, test_with_prot_embeddings.parquet


## Compute ChemBERTa Embeddings for the Test Set

In this cell we compute **ligand embeddings using ChemBERTa**.

The procedure consists of:

1. Cleaning and validating SMILES strings
2. Tokenization using the ChemBERTa tokenizer
3. Forward pass through the transformer model
4. Mean pooling of token embeddings
5. L2 normalization of the resulting vectors

Each molecule is represented by a **768-dimensional embedding vector**.

In [ ]:
# Model Parameters
CHEM_SMILES_COL = "canonical_smiles"
CHEM_MODEL_NAME = "seyonec/ChemBERTa-zinc-base-v1"  # hidden size = 768
CHEM_DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
CHEM_BATCH_SIZE = 64
CHEM_MAX_LENGTH = 514
CHEM_POOLING    = "mean"   # "mean" or "cls"
CHEM_NORMALIZE  = True
EPS = 1e-12

# Reuse ChemBERTa helper function defined in the training embedding cell.
required_helpers = ["embed_batch"]
missing_helpers = [name for name in required_helpers if name not in globals()]
if missing_helpers:
    raise NameError(
        "Run the ChemBERTa training embedding cell first; missing helpers: "
        + ", ".join(missing_helpers)
    )

# load tokenizer/model
if "chem_tokenizer" not in globals() or "chem_model" not in globals():
    print(f"Load ChemBERTa: {CHEM_MODEL_NAME} (device={CHEM_DEVICE})")
    chem_tokenizer = AutoTokenizer.from_pretrained(CHEM_MODEL_NAME)
    chem_model = AutoModel.from_pretrained(CHEM_MODEL_NAME).to(CHEM_DEVICE).eval()

H = chem_model.config.hidden_size  # 768

smiles_raw = df_test[CHEM_SMILES_COL].tolist()
N = len(smiles_raw)

smiles_str = [
    ("" if (s is None or (isinstance(s, float) and np.isnan(s))) else str(s))
    for s in smiles_raw
]
is_valid = [(s.strip() != "") and (s.strip().lower() != "nan") for s in smiles_str]
valid_idx = [i for i, ok in enumerate(is_valid) if ok]
valid_texts = [smiles_str[i].strip() for i in valid_idx]

print(f"Total row test: {N} | SMILES: {len(valid_texts)} | empty SMILES: {N-len(valid_texts)}")

vecs_valid = []
for i in tqdm(range(0, len(valid_texts), CHEM_BATCH_SIZE), desc="ChemBERTa embeddings (test)"):
    batch = valid_texts[i:i + CHEM_BATCH_SIZE]
    vecs_valid.append(
        embed_batch(
            batch,
            chem_tokenizer,
            chem_model,
            CHEM_DEVICE,
            CHEM_MAX_LENGTH,
            pooling=CHEM_POOLING
        )
    )

X_valid = np.vstack(vecs_valid) if len(vecs_valid) else np.zeros((0, H), dtype=np.float32)

X = np.zeros((N, H), dtype=np.float32)
for j, i_orig in enumerate(valid_idx):
    X[i_orig, :] = X_valid[j, :]

if CHEM_NORMALIZE:
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    X = np.where(norms > 0, X / (norms + EPS), X)

print("Test ChemBERTa shape:", X.shape, "mean norm:", float(np.mean(np.linalg.norm(X, axis=1))))

df_test["ligand_embeddings"] = X.tolist()

np.save("test_ligand_embeddings.npy", X)
df_test.to_parquet("test_set_with_embeddings.parquet", index=False, engine="pyarrow")
print("Saved: test_ligand_embeddings.npy, test_set_with_embeddings.parquet")


Carico ChemBERTa: seyonec/ChemBERTa-zinc-base-v1 (device=cpu)
Total row test: 177 | SMILES: 177 | empty SMILES: 0


ChemBERTa embeddings (test):   0%|          | 0/3 [00:00<?, ?it/s]

Test ChemBERTa shape: (177, 768) mean norm: 1.0
Saved: test_ligand_embeddings.npy, test_set_with_embeddings.parquet


### Preparation of the final training and test matrices

In this step we construct the feature matrices used by XGB regressor.

For each protein–ligand pair we concatenate:
- the **ligand embedding**
- the **protein embedding**

This produces the final input vector for the model.

We then build:

- **X_train_full**: feature matrix for the training set  
- **y_train_full**: target values (-log10(Koff)) for the training set  
- **X_test**: feature matrix for the test set  
- **y_test**: target values for the test set

Finally, we replace potential NaN or infinite values to ensure numerical stability during training.

In [17]:
# Check required columns
for name, df in [("df_train", df_train), ("df_test", df_test)]:
    for c in ["prot_embeddings", "ligand_embeddings", "koff"]:
        if c not in df.columns:
            raise KeyError(f"Column '{c}' missing in {name}")
# Build feature matrices
X_train_full = np.concatenate([
    np.vstack(df_train["ligand_embeddings"].values).astype(np.float32),
    np.vstack(df_train["prot_embeddings"].values).astype(np.float32)
], axis=1)

y_train_full = df_train["koff"].to_numpy(dtype=np.float32)
X_test = np.concatenate([
    np.vstack(df_test["ligand_embeddings"].values).astype(np.float32),
    np.vstack(df_test["prot_embeddings"].values).astype(np.float32)
], axis=1)

y_test = df_test["koff"].to_numpy(dtype=np.float32)
# Safety replacement
X_train_full = np.nan_to_num(X_train_full, nan=0.0, posinf=1e6, neginf=-1e6)
X_test = np.nan_to_num(X_test, nan=0.0, posinf=1e6, neginf=-1e6)
print("Train shape:", X_train_full.shape)
print("Test shape:", X_test.shape)

Train shape: (709, 1792)
Test shape: (177, 1792)


###  Final Model Training and Test Evaluation (XGBoost)

After completing hyperparameter optimization, the final XGBoost model was trained on the entire training dataset using the best configuration identified in the previous step. This ensures that the model leverages all available training data before final evaluation.

The trained model was then serialized and saved to disk for future use, along with the corresponding hyperparameter configuration to guarantee reproducibility.

Predictions were generated for both the training set and the independent test set. Model performance was evaluated using three metrics: Root Mean Squared Error (RMSE), coefficient of determination (\(R^2\)), and Pearson correlation coefficient.

Evaluating both training and test performance allows for assessing potential overfitting and the generalization capability of the final model on unseen data.

In [18]:
# TRAIN FINAL MODEL ON FULL TRAIN

final_model = xgb.XGBRegressor(
    **best_xgb_params,
    objective="reg:squarederror",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

final_model.fit(X_train_full, y_train_full)

print("Model trained on full training set.")

# SAVE MODEL

MODEL_PATH = "final_xgb_model.pkl"
with open(MODEL_PATH, "wb") as f:
    pickle.dump(final_model, f)

print(f"Model saved to: {MODEL_PATH}")

# save params
CONFIG_PATH = "xgb_config.json"
with open(CONFIG_PATH, "w") as f:
    json.dump(best_xgb_params, f)

print(f"Config saved to: {CONFIG_PATH}")

# METRICS

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def pearson_corr(y_true, y_pred):
    r, _ = pearsonr(y_true, y_pred)
    return r

# PREDICTIONS

train_preds = final_model.predict(X_train_full)
test_preds  = final_model.predict(X_test)

# EVALUATION

train_r2 = r2_score(y_train_full, train_preds)
train_rmse = rmse(y_train_full, train_preds)
train_pearson = pearson_corr(y_train_full, train_preds)

test_r2 = r2_score(y_test, test_preds)
test_rmse = rmse(y_test, test_preds)
test_pearson = pearson_corr(y_test, test_preds)

print("\nFinal performance (XGBoost)")
print(f"TRAIN -> R2: {train_r2:.3f} | RMSE: {train_rmse:.3f} | Pearson r: {train_pearson:.3f}")
print(f"TEST  -> R2: {test_r2:.3f} | RMSE: {test_rmse:.3f} | Pearson r: {test_pearson:.3f}")

Model trained on full training set.
Model saved to: final_xgb_model.pkl
Config saved to: xgb_config.json

Final performance (XGBoost)
TRAIN -> R2: 0.927 | RMSE: 0.277 | Pearson r: 0.975
TEST  -> R2: 0.463 | RMSE: 0.728 | Pearson r: 0.682
